# Introduction to Ray: Core, Data, Train, Tune & Serve

Ray is a framework for scaling Python applications from a laptop to a cluster.
It has a small core (tasks & actors) and a set of higher-level libraries built on top of it:

- **Ray Core** — parallel/distributed Python with `@ray.remote`
- **Ray Data** — distributed data loading & preprocessing
- **Ray Train** — distributed model training
- **Ray Tune** — hyperparameter search
- **Ray Serve** — model deployment / online inference

In this notebook we'll walk through each of these using a tiny example: classifying
handwritten digits (the classic `sklearn` digits dataset, 8x8 images, 10 classes).

In [ ]:
import ray

ray.init(ignore_reinit_error=True)
ray.cluster_resources()

### Ray Dashboard

`ray.init()` starts a local dashboard at **http://127.0.0.1:8265**.

| Tab | What it shows |
|---|---|
| **Cluster** | Node-level CPU / memory usage |
| **Jobs** | Status of each submitted job; click to drill into tasks |
| **Tasks** | Live task graph — running, pending, failed |
| **Actors** | Live actors, method call counts, memory |
| **Logs** | Per-task and per-actor stdout/stderr; filter by Job ID |
| **Metrics** | Time-series graphs (needs Prometheus for full view) |

To trace a specific piece of code: copy the **Job ID** from the Jobs tab, then filter Logs by it.


## 1. Ray Core — Tasks and Remote Functions

The core building block of Ray is the **remote function** (a *task*). Decorating a
function with `@ray.remote` turns calls to it into asynchronous jobs that Ray schedules
across available CPUs/GPUs.

- `.remote()` schedules the task and immediately returns a *future* (an `ObjectRef`)
- `ray.get()` blocks until the result is ready and returns the value

Below, 8 "slow" tasks that each take ~1 second run in parallel instead of sequentially.

In [ ]:
import time


@ray.remote
def slow_square(x):
    time.sleep(1)
    return x * x


start = time.time()
futures = [slow_square.remote(i) for i in range(8)]
results = ray.get(futures)
elapsed = time.time() - start
print(futures)
print("results:", results)
print(f"took {elapsed:.1f}s (would take ~8s if run one-by-one)")

# sequential version
start = time.time()

results = []

for i in range(8):
    ref = slow_square.remote(i)
    results.append(ray.get(ref))

elapsed = time.time() - start
print("results:", results)
print(f"took {elapsed:.1f}s (would take ~8s if run one-by-one)")

### Futures and ObjectRefs

`.remote()` immediately returns an **`ObjectRef`** — a handle to a future result stored in Ray's
distributed object store. You can pass `ObjectRef`s directly to other `.remote()` calls;
Ray builds the dependency graph automatically and waits only as long as needed.

- **`ray.get(refs)`** — blocks until *all* results are ready
- **`ray.wait(refs, num_returns=k)`** — returns as soon as *k* results are ready, leaving the rest pending — useful when tasks have variable runtimes


In [ ]:
# ObjectRef — the actual object returned by .remote()
ref = slow_square.remote(5)
print("ObjectRef:", ref)          # ObjectRef(abc123...)
print("value:    ", ray.get(ref))  # 25

# ray.wait() — process results as they finish instead of waiting for all
futures = [slow_square.remote(i) for i in range(6)]
order = []
remaining = list(futures)
while remaining:
    done, remaining = ray.wait(remaining, num_returns=1) 
    order.append(ray.get(done[0]))

print("results in completion order:", order)


### Scheduler and CPU Allocation

Ray's scheduler assigns tasks to workers based on **requested resources**. By default each
task claims 1 CPU. Override with `@ray.remote(num_cpus=N)`:

- `num_cpus=2` → at most 4 concurrent tasks on an 8-CPU node (limits parallelism for heavy work)
- `num_cpus=0.5` → up to 16 concurrent tasks (pack lightweight tasks tighter)
- `num_gpus=1` → task waits until a GPU slot is free

The scheduler never over-commits: if resources are unavailable the task queues until they are.


In [ ]:
@ray.remote(num_cpus=1)
def light_task(n):
    time.sleep(0.5)
    return n

@ray.remote(num_cpus=2)
def heavy_task(n):
    time.sleep(0.5)
    return n

available_cpus = int(ray.cluster_resources().get("CPU", 1))
print("cluster CPUs:", available_cpus)

# light_task: all 8 run in one round (~0.5 s)
t0 = time.time()
ray.get([light_task.remote(i) for i in range(8)])
print(f"light_task (1 CPU each): {time.time()-t0:.1f}s")

# heavy_task: only 4 fit at once → 2 rounds (~1.0 s)
t0 = time.time()
ray.get([heavy_task.remote(i) for i in range(8)])
print(f"heavy_task (2 CPU each): {time.time()-t0:.1f}s")


### Ray Actors — Stateful Workers

A **remote function** is stateless — each call is independent. An **actor** is a long-lived stateful object on the cluster. Method calls are queued and executed serially, preserving state across calls.

Actors are useful for:
- Holding model weights in memory for repeated inference
- Accumulating statistics across streaming batches
- Building parameter servers or shared caches

In [ ]:
import random


@ray.remote
class RunningStats:
    """Tracks running mean and std over streaming batches (Welford's algorithm)."""

    def __init__(self):
        self.n = 0
        self.mean = 0.0
        self._M2 = 0.0

    def update(self, values):
        for x in values:
            self.n += 1
            delta = x - self.mean
            self.mean += delta / self.n
            self._M2 += delta * (x - self.mean)

    def stats(self):
        std = (self._M2 / self.n) ** 0.5 if self.n > 1 else 0.0
        return {"n": self.n, "mean": round(self.mean, 4), "std": round(std, 4)}


# actors are created with .remote() and live on the cluster until killed
stats_actor = RunningStats.remote()

random.seed(42)
batches = [[random.gauss(5.0, 2.0) for _ in range(250)] for _ in range(4)]

for batch in batches:
    ray.get(stats_actor.update.remote(batch))  # state accumulates across calls

print("Stats over 1000 samples:", ray.get(stats_actor.stats.remote()))
# expected: mean ≈ 5.0, std ≈ 2.0


### Memory and the Object Store

Results of `.remote()` calls live in Ray's **shared-memory object store** (Apache Plasma).
Workers on the same node read objects zero-copy via shared memory; cross-node transfers go over the network.

`ray.put(obj)` explicitly places a large object in the store **once** and returns a ref.
Pass that ref to many tasks instead of re-serialising the object on every call — critical for
large arrays or model weights shared across many workers.


In [ ]:
import numpy as np

big_array = np.random.rand(1000, 64).astype(np.float32)

@ray.remote
def row_sum(data, start, end):
    return float(data[start:end].sum())

# Without ray.put: big_array is serialised separately for each of the 4 tasks
futures_no_put = [row_sum.remote(big_array, i*250, (i+1)*250) for i in range(4)]

# With ray.put: serialised ONCE, all 4 tasks share the same object store entry
data_ref = ray.put(big_array)
futures_put = [row_sum.remote(data_ref, i*250, (i+1)*250) for i in range(4)]

print("total (no put):", round(sum(ray.get(futures_no_put)), 4))
print("total (put):   ", round(sum(ray.get(futures_put)),    4))  # same result
print("ObjectRef from ray.put:", data_ref)


## 2. Ray Data — Distributed Data Loading & Preprocessing

`ray.data.Dataset` is a distributed collection of data that can be transformed with
operations like `map_batches`. Transformations are **lazy** and run in parallel across
the cluster, which is what makes Ray Data useful for preprocessing large datasets
before training.

Here we load the digits dataset into a Dataset and normalize pixel values from `[0, 16]`
to `[0, 1]` using `map_batches`.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)
X = X.astype(np.float32)
y = y.astype(np.int64)

ds = ray.data.from_numpy(X)
print(ds.schema())
print("num rows:", ds.count())

In [ ]:
import math

num_blocks = ds.num_blocks()
num_rows = ds.count()
print(f"blocks (partitions) : {num_blocks}")
print(f"total rows          : {num_rows}")

# iter_batches mirrors what map_batches sees — no batch_size means one batch per block
print("\n── batch_size=None → one batch per block ──")
for i, b in enumerate(ds.iter_batches(batch_size=None, batch_format="numpy")):
    print(f"  batch {i:2d}: {len(b['data'])} rows")

print("\n── explicit batch_size → rows chunked across blocks ──")
for bsz in [1024, 512]:
    n = sum(1 for _ in ds.iter_batches(batch_size=bsz, batch_format="numpy"))
    print(f"  batch_size={bsz:4d} → {n} batches  (⌈{num_rows}/{bsz}⌉ = {math.ceil(num_rows / bsz)})")

In [ ]:
@ray.remote
class _RankCounter:
    def __init__(self): self._n = 0
    def next(self):
        r = self._n; self._n += 1; return r

_counter = _RankCounter.remote()
WORLD_SIZE = 2  # number of parallel actors = "world_size"

class Normalizer:
    def __init__(self):
        # each actor gets a unique rank when it starts up
        self.rank = ray.get(_counter.next.remote())

    def __call__(self, batch):
        print(f"  rank={self.rank}  world_size={WORLD_SIZE}  batch_size={len(batch['data'])}")
        batch["data"] = batch["data"] / 16.0
        return batch


normalized_ds = ds.map_batches(Normalizer, batch_format="numpy", concurrency=WORLD_SIZE)

# .take() triggers execution of the lazy pipeline on a small sample
sample = normalized_ds.take(1)[0]
print("min/max after normalization:", sample["data"].min(), sample["data"].max())

## 3. Ray Train — Distributed Training (DDP + AllReduce)

Ray Train wraps PyTorch **DistributedDataParallel (DDP)** across multiple workers.
Here's what happens under the hood each training step:

1. **Data sharding** — each worker receives a non-overlapping slice of the data (`rank::world_size`)
2. **Forward pass** — each worker independently computes the loss on its own shard
3. **Backward pass** — each worker computes local gradients
4. **AllReduce** — DDP averages gradients across *all* workers (ring-AllReduce); every worker
   ends up with the same averaged gradient
5. **Weight update** — each worker applies the identical averaged gradient, keeping all model
   copies in sync without a central parameter server

Net effect: training with an effective batch size of `world_size × local_batch`, distributed
across workers. `ray.train.torch.prepare_model(model)` wraps the model in DDP and handles
the AllReduce communication for you.


In [ ]:
import torch
from torch import nn
from sklearn.model_selection import train_test_split


class DigitClassifier(nn.Module):
    def __init__(self, hidden_size: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 10),
        )

    def forward(self, x):
        return self.net(x)


def get_train_test_tensors():
    X_norm = X / 16.0
    X_train, X_test, y_train, y_test = train_test_split(
        X_norm, y, test_size=0.2, random_state=42
    )
    return (
        torch.from_numpy(X_train),
        torch.from_numpy(X_test),
        torch.from_numpy(y_train),
        torch.from_numpy(y_test),
    )

In [ ]:
import os
import tempfile

import ray.train
from ray.train import Checkpoint, ScalingConfig
from ray.train.torch import TorchTrainer


def train_loop_per_worker(config):
    X_train, X_test, y_train, y_test = get_train_test_tensors()

    # shard training data so each worker sees a different subset
    rank = ray.train.get_context().get_world_rank()
    world_size = ray.train.get_context().get_world_size()
    X_train = X_train[rank::world_size]
    y_train = y_train[rank::world_size]

    model = DigitClassifier(hidden_size=config["hidden_size"])
    model = ray.train.torch.prepare_model(model)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(config["epochs"]):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            accuracy = (model(X_test).argmax(dim=1) == y_test).float().mean().item()

        metrics = {"epoch": epoch, "loss": loss.item(), "accuracy": accuracy}

        if epoch == config["epochs"] - 1:
            state_dict = (
                model.module.state_dict() if hasattr(model, "module") else model.state_dict()
            )
            with tempfile.TemporaryDirectory() as tmpdir:
                torch.save(
                    {"model_state_dict": state_dict, "hidden_size": config["hidden_size"]},
                    os.path.join(tmpdir, "model.pt"),
                )
                ray.train.report(metrics, checkpoint=Checkpoint.from_directory(tmpdir))
        else:
            ray.train.report(metrics)


In [ ]:
trainer = TorchTrainer(
    train_loop_per_worker,
    train_loop_config={"hidden_size": 64, "lr": 1e-2, "epochs": 30},
    scaling_config=ScalingConfig(num_workers=2, use_gpu=False),
)

train_result = trainer.fit()
print("final metrics:", train_result.metrics)

### Parallel Model Inference with an Actor Pool

Spin up a pool of inference actors — each holding a loaded model — and scatter prediction
requests across them. Model weights stay in memory between requests (no repeated loading),
and all CPUs are kept busy.


In [ ]:
@ray.remote
class InferenceWorker:
    def __init__(self, state_dict, hidden_size):
        self.model = DigitClassifier(hidden_size=hidden_size)
        self.model.load_state_dict(state_dict)
        self.model.eval()

    def predict(self, X_batch: np.ndarray) -> list:
        with torch.no_grad():
            t = torch.from_numpy(X_batch.astype(np.float32))
            return self.model(t).argmax(dim=1).tolist()


# load weights from the Ray Train checkpoint
with train_result.checkpoint.as_directory() as ckpt_dir:
    ckpt = torch.load(os.path.join(ckpt_dir, "model.pt"), map_location="cpu")

num_workers = 4
workers = [
    InferenceWorker.remote(ckpt["model_state_dict"], ckpt["hidden_size"])
    for _ in range(num_workers)
]

# distribute full dataset across workers and run in parallel
X_all = (X / 16.0).astype(np.float32)
chunks = np.array_split(X_all, num_workers)

futures = [w.predict.remote(chunk) for w, chunk in zip(workers, chunks)]
all_preds = [p for batch in ray.get(futures) for p in batch]

accuracy = sum(p == t for p, t in zip(all_preds, y)) / len(y)
print(f"Parallel inference accuracy: {accuracy:.4f}  ({len(all_preds)} samples, {num_workers} workers)")


## 4. Ray Tune — Hyperparameter Search

Ray Tune runs many training trials in parallel, each with a different combination of
hyperparameters, and reports which one performs best. You give it a training function
and a *search space*; Tune handles scheduling the trials across the cluster.

Here we search over the hidden layer size and the learning rate for the same digit
classifier, training each trial for a few epochs.

In [ ]:
from ray import tune

def train_digit_classifier(config):
    X_train, X_test, y_train, y_test = get_train_test_tensors()

    model = DigitClassifier(hidden_size=config["hidden_size"])
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(15):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        accuracy = (model(X_test).argmax(dim=1) == y_test).float().mean().item()

    tune.report({"accuracy": accuracy})


search_space = {
    "hidden_size": tune.choice([32, 64, 128]),
    "lr": tune.loguniform(1e-3, 1e-1),
}

tuner = tune.Tuner(
    train_digit_classifier,
    param_space=search_space,
    tune_config=tune.TuneConfig(metric="accuracy", mode="max", num_samples=8),
)

tune_results = tuner.fit()
tune_best_result = tune_results.get_best_result()

print("best config:", tune_best_result.config)
print("best accuracy:", tune_best_result.metrics["accuracy"])

## 5. Ray Serve — Model Deployment

Ray Serve turns a Python class into a scalable HTTP endpoint. We load the weights from
the model we trained with Ray Train into the deployment, start it, and then send it some
requests.

In [ ]:
from starlette.requests import Request
from starlette.responses import JSONResponse
from ray import serve

# Load the best checkpoint's weights
with train_result.checkpoint.as_directory() as ckpt_dir:
    checkpoint = torch.load(os.path.join(ckpt_dir, "model.pt"), map_location="cpu")


@serve.deployment
class DigitClassifierDeployment:
    def __init__(self, state_dict, hidden_size):
        self.model = DigitClassifier(hidden_size=hidden_size)
        self.model.load_state_dict(state_dict)
        self.model.eval()

    async def __call__(self, request: Request):
        if request.method == "GET":
            return JSONResponse({"status": "ok", "usage": "POST {\"features\": [64 floats]}"})

        payload = await request.json()
        features = torch.tensor([payload["features"]], dtype=torch.float32)
        with torch.no_grad():
            prediction = int(self.model(features).argmax(dim=1).item())
        return {"prediction": prediction}


app = DigitClassifierDeployment.bind(checkpoint["model_state_dict"], checkpoint["hidden_size"])
serve.run(app)

In [ ]:
import requests

for i in range(5):
    sample = (X[i] / 16.0).tolist()
    response = requests.post("http://127.0.0.1:8000/", json={"features": sample})
    print("predicted:", response.json()["prediction"], "| true label:", y[i])

## Wrap-up & Cleanup

- **Ray tasks** (`@ray.remote` function): stateless parallel work
- **ObjectRefs / Futures**: handles to results in the object store; pass between tasks without materialising
- **`ray.get()`** / **`ray.wait()`**: block for all / unblock as results arrive
- **Scheduler & CPU allocation**: `@ray.remote(num_cpus=N)` controls concurrency and resource packing
- **Dashboard** (`127.0.0.1:8265`): Cluster, Jobs, Tasks, Actors, Logs, Metrics
- **Ray actors** (`@ray.remote` class): stateful long-lived workers
- **Object store / `ray.put()`**: shared memory for large objects; avoid repeated serialisation
- **Ray Data**: `from_numpy(...).map_batches(fn)` for distributed preprocessing
- **Ray Train (DDP + AllReduce)**: `TorchTrainer` with data sharding; `prepare_model` handles gradient sync
- **Parallel inference**: actor pool scatters requests across workers with model weights in memory
- **Ray Tune**: `tune.Tuner(fn, param_space=...)` for hyperparameter search
- **Ray Serve**: `@serve.deployment` + `serve.run(app)` for online inference
- **Debugging**: `RayTaskError`, `inspect_serializability`, `ray.timeline()`, dashboard Logs

Shut everything down when you're done:


In [ ]:
serve.shutdown()
ray.shutdown()